# Colab Motion Pipeline: GPU-Accelerated 2D Subpixel Motion Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yutotuy0209-droid/colab-motion-pipeline/blob/main/notebooks/colab_motion_analysis_demo.ipynb)

This notebook demonstrates **GPU-accelerated Phase-Only Correlation (POC)** for 2D subpixel displacement estimation and linear error identification ($L_{\text{meas}} = \alpha L_{\text{cmd}} + \beta + \epsilon$).

In [ ]:
# Step 1: Environment Check & Setup
import os
import sys

# Clone repository if running in Google Colab
if 'google.colab' in sys.modules:
    if not os.path.exists('colab-motion-pipeline'):
        !git clone https://github.com/yutotuy0209-droid/colab-motion-pipeline.git
        %cd colab-motion-pipeline
        sys.path.append(os.getcwd())

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU mode")

In [ ]:
# Step 2: Import Core Modules
import numpy as np
import matplotlib.pyplot as plt

from src.poc_core import compute_displacement
from src.evaluation import fit_linear_motion_model
from src.synthetic_data import generate_motion_sequence

print("Modules successfully loaded!")

In [ ]:
# Step 3: Generate Benchmark Sequence (Known Subpixel Displacements)
num_frames = 25
step_dx = 0.20  # 0.20 px/step
step_dy = 0.08  # 0.08 px/step

print(f"Generating {num_frames} synthetic grid frames...")
frames, true_dx, true_dy = generate_motion_sequence(
    num_frames=num_frames,
    step_dx=step_dx,
    step_dy=step_dy,
    height=512,
    width=512
)

# Visualize reference frame
plt.figure(figsize=(6, 6))
plt.imshow(frames[0], cmap='gray')
plt.title("Reference Frame (Frame 0) - Dot Grid Pattern")
plt.colorbar(fraction=0.046, pad=0.04)
plt.show()

In [ ]:
# Step 4: Batch Displacement Estimation with POC
import time

ref_frame = frames[0]
meas_dx = []
meas_dy = []
peak_values = []

start_time = time.time()
for i in range(num_frames):
    dx, dy, peak = compute_displacement(ref_frame, frames[i], use_gpu=True)
    meas_dx.append(dx)
    meas_dy.append(dy)
    peak_values.append(peak)

elapsed = (time.time() - start_time) * 1000.0
fps = num_frames / (elapsed / 1000.0)

meas_dx = np.array(meas_dx)
meas_dy = np.array(meas_dy)

print(f"Processed {num_frames} frames in {elapsed:.2f} ms ({fps:.1f} FPS)")

In [ ]:
# Step 5: Linear Motion Model Fitting & Error Analysis
eval_x = fit_linear_motion_model(true_dx, meas_dx)
eval_y = fit_linear_motion_model(true_dy, meas_dy)

print("=== X-Axis Evaluation ===")
print(f"  Scale Slope (alpha): {eval_x['alpha']:.5f} (Ideal: 1.00000)")
print(f"  Offset (beta):       {eval_x['beta']:.5f} px")
print(f"  RMSE:                {eval_x['rmse']:.5f} px")
print(f"  3-Sigma Error:       {eval_x['three_sigma']:.5f} px")
print(f"  R^2:                 {eval_x['r_squared']:.6f}")

print("\n=== Y-Axis Evaluation ===")
print(f"  Scale Slope (alpha): {eval_y['alpha']:.5f} (Ideal: 1.00000)")
print(f"  Offset (beta):       {eval_y['beta']:.5f} px")
print(f"  RMSE:                {eval_y['rmse']:.5f} px")
print(f"  3-Sigma Error:       {eval_y['three_sigma']:.5f} px")
print(f"  R^2:                 {eval_y['r_squared']:.6f}")

In [ ]:
# Step 6: Professional Academic Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Command vs Measurement
axes[0].plot(true_dx, meas_dx, 'o-', color='#2563EB', label='X-Measured')
axes[0].plot(true_dx, eval_x['y_pred'], '--', color='#163B63', label=f'Fit (alpha={eval_x["alpha"]:.4f})')
axes[0].plot(true_dy, meas_dy, 's-', color='#10B981', label='Y-Measured')
axes[0].plot(true_dy, eval_y['y_pred'], '--', color='#065F46', label=f'Fit (alpha={eval_y["alpha"]:.4f})')
axes[0].set_title("Commanded vs Measured Displacement", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Ground Truth Displacement [pixel]", fontsize=11)
axes[0].set_ylabel("Measured POC Displacement [pixel]", fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(fontsize=10)

# Right: Residual Error Distribution
axes[1].plot(range(num_frames), eval_x['residuals'], 'o-', color='#EF4444', label=f'X Residuals (3σ={eval_x["three_sigma"]:.4f}px)')
axes[1].plot(range(num_frames), eval_y['residuals'], 's-', color='#F59E0B', label=f'Y Residuals (3σ={eval_y["three_sigma"]:.4f}px)')
axes[1].axhline(0, color='gray', linestyle=':')
axes[1].set_title("Residual Measurement Noise (Subpixel Scale)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Frame Index", fontsize=11)
axes[1].set_ylabel("Residual Error [pixel]", fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig("motion_analysis_result.png", dpi=300)
plt.show()
print("Figure saved as motion_analysis_result.png")